In [1]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import seaborn as sns
from matplotlib.image import imread
from PIL import Image
import tensorflow as tf
np.random.seed(1337)
import gc
from tensorflow.keras.utils import to_categorical

import glob

from tensorflow.keras import layers
from keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.models import Sequential,Model
from tensorflow.keras.layers import Input,Reshape,Multiply, Dropout, AveragePooling2D,Flatten, Dense, Conv2D,MaxPool2D, MaxPooling2D, BatchNormalization,concatenate,UpSampling2D
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')


img_size = 128
dataset = os.listdir("music_dataset_spectro_full/train")
labels = dataset
print(labels)

['Accordion', 'Acoustic_Guitar', 'Banjo', 'Bass_Guitar', 'Clarinet', 'cowbell', 'Dobro', 'Drum_set', 'Electric_Guitar', 'flute', 'Harmonium', 'Horn', 'Keyboard', 'Mandolin', 'Organ', 'Piano', 'Saxophone', 'Shakers', 'Tambourine', 'Trombone', 'Trumpet', 'Ukulele', 'vibraphone', 'Violin']


In [2]:
def get_mixes_array(data_dir):
    data = []
    path = os.path.join(data_dir+"full_mix")
    for img in os.listdir(data_dir+"full_mix"):
        for stem in os.listdir(data_dir+"stems"):
            if img.split("_")[0] == stem.split("_")[0]:
                temp=stem.split("_",1)
                label = temp[1].split(".")[0]
                class_num = labels.index(label)
                try:
                    img_arr = cv2.imread(os.path.join(path,img),0)
                    resized_arr = cv2.resize(img_arr, (img_size, img_size))
                    data.append([resized_arr,class_num])
                    gc.collect()
                except Exception as e:
                    print(e)
    return np.array(data,dtype="object") 


In [3]:
def get_stems_array(data_dir):
    data = []
    path = os.path.join(data_dir)
    for img in os.listdir(data_dir):
            try:
                img_arr = cv2.imread(os.path.join(path,img),0)
                resized_arr = cv2.resize(img_arr, (img_size, img_size))
                data.append([resized_arr])
                gc.collect()
            except Exception as e:
                print(e)
    return np.array(data,dtype="float32") 

In [4]:
x_train = get_mixes_array("music_seperation_dataset/train/")

y_train = get_stems_array("music_seperation_dataset/train/stems")

x_test = get_mixes_array("music_seperation_dataset/test/")

y_test = get_stems_array("music_seperation_dataset/test/stems")

x_valid = get_mixes_array("music_seperation_dataset/valid/")

y_valid = get_stems_array("music_seperation_dataset/valid/stems")



In [5]:
x_train_mix=[]
x_train_label=[]

x_test_mix=[]
x_test_label=[]

x_valid_mix=[]
x_valid_label=[]

for feature, label in x_train:
    x_train_mix.append(feature)
    x_train_label.append(label)

for feature, label in x_test:
    x_test_mix.append(feature)
    x_test_label.append(label)

for feature, label in x_valid:
    x_valid_mix.append(feature)
    x_valid_label.append(label)

del x_train
del x_test
del x_valid

In [6]:
gc.collect()
x_train_mix = np.array(x_train_mix)/255
gc.collect()
x_test_mix = np.array(x_test_mix)/255
gc.collect()
x_valid_mix = np.array(x_valid_mix)/255
gc.collect()
y_train = np.array(y_train)/255
gc.collect()
y_test = np.array(y_test)/255
gc.collect()
y_valid = np.array(y_valid)/255
gc.collect()


0

In [7]:
x_train_mix = x_train_mix.reshape(-1, img_size, img_size, 1)
y_train = y_train.reshape(-1, img_size, img_size, 1)

x_valid_mix = x_valid_mix.reshape(-1, img_size, img_size, 1)
y_valid = y_valid.reshape(-1, img_size, img_size, 1)

x_test_mix = x_test_mix.reshape(-1, img_size, img_size, 1)
y_test = y_test.reshape(-1, img_size, img_size, 1)


In [8]:
x_train_label = np.array(x_train_label)
x_train_label = to_categorical(x_train_label)

x_test_label = np.array(x_test_label)
x_test_label = to_categorical(x_test_label)

x_valid_label = np.array(x_valid_label)
x_valid_label = to_categorical(x_valid_label)

In [20]:
print(x_valid_label)

[[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.

In [30]:
num_classes = 13
def Unet():
    inputs =  Input(shape=(None,None,1))
    label_input = Input(shape=(num_classes,))

    conv1 = Conv2D(16, (3,3), activation = 'relu', padding='same')(inputs)
    conv1 = BatchNormalization()(conv1)
    conv1 = Conv2D(16, (3,3), activation = 'relu', padding='same')(conv1)
    conv1 = BatchNormalization()(conv1)
    pool1 = MaxPool2D((2,2))(conv1)

    conv2 = Conv2D(32, (3,3), activation = 'relu', padding='same')(pool1)
    conv2 = BatchNormalization()(conv2)
    conv2 = Conv2D(32, (3,3), activation = 'relu', padding='same')(conv2)
    conv2 = BatchNormalization()(conv2)
    pool2 = MaxPool2D((2,2))(conv2)

    conv3 = Conv2D(64, (3,3), activation = 'relu', padding='same')(pool2)
    conv3 = BatchNormalization()(conv3)
    conv3 = Conv2D(64, (3,3), activation = 'relu', padding='same')(conv3)
    conv3 = BatchNormalization()(conv3)
    pool3 = MaxPool2D((2,2))(conv3)

    conv4 = Conv2D(128, (3,3), activation = 'relu', padding='same')(pool3)
    conv4 = BatchNormalization()(conv4)
    conv4 = Conv2D(128, (3,3), activation = 'relu', padding='same')(conv4)
    conv4 = BatchNormalization()(conv4)
    pool4 = MaxPool2D((2,2))(conv4)

    conv5 = Conv2D(256, (3,3), activation = 'relu', padding='same')(pool4)
    conv5 = BatchNormalization()(conv5)
    conv5 = Conv2D(256, (3,3), activation = 'relu', padding='same')(conv5)
    conv5 = BatchNormalization()(conv5)
    
    label_lay = Dense(256,activation="relu")(label_input)
    label_lay = Reshape((1,1,256))(label_lay)
    multi_bottle = Multiply()([conv5,label_lay])
    
    goUp1 = UpSampling2D((2,2))(multi_bottle)
    goUp1 = Conv2D(128,(3,3),padding='same',activation = 'relu',)(goUp1)
    goUp1 = BatchNormalization()(goUp1)
    goUp1 = concatenate([goUp1,conv4])
    conv6 = Conv2D(128, (3,3), activation = 'relu', padding='same')(goUp1)
    conv6 = BatchNormalization()(conv6)
    conv6 = Conv2D(128, (3,3), activation = 'relu', padding='same')(conv6)
    conv6 = BatchNormalization()(conv6)

    goUp2 = UpSampling2D((2,2))(conv6)
    goUp2 = Conv2D(64,(3,3),padding='same',activation = 'relu',)(goUp2)
    goUp2 = BatchNormalization()(goUp2)
    goUp2 = concatenate([goUp2,conv3])
    conv7 = Conv2D(64, (3,3), activation = 'relu', padding='same')(goUp2)
    conv7 = BatchNormalization()(conv7)
    conv7 = Conv2D(64, (3,3), activation = 'relu', padding='same')(conv7)
    conv7 = BatchNormalization()(conv7)

    goUp3 = UpSampling2D((2,2))(conv7)
    goUp3 = Conv2D(32,(3,3),padding='same',activation = 'relu',)(goUp3)
    goUp3 = concatenate([goUp3,conv2])
    goUp3 = BatchNormalization()(goUp3)
    conv8 = Conv2D(32, (3,3), activation = 'relu', padding='same')(goUp3)
    conv8 = BatchNormalization()(conv8)
    conv8 = Conv2D(32, (3,3), activation = 'relu', padding='same',)(conv8)
    conv8 = BatchNormalization()(conv8)

    goUp4 = UpSampling2D((2,2))(conv8)
    goUp4 = Conv2D(16,(3,3),padding='same',activation = 'relu',)(goUp4)
    goUp4 = BatchNormalization()(goUp4)
    goUp4 = concatenate([goUp4,conv1])
    conv9 = Conv2D(16, (3,3), activation = 'relu', padding='same')(goUp4)
    conv9 = BatchNormalization()(conv9)
    conv9 = Conv2D(16, (3,3), activation = 'relu', padding='same')(conv9)
    conv9 = BatchNormalization()(conv9)



    outputs = Conv2D(1,(1,1),activation="sigmoid")(conv9)

    model = Model(inputs=[inputs,label_input],outputs=[outputs])
    return model

model = Unet()
model.compile(
              optimizer = 'adam', loss = 'mse',
              metrics = ['mae']
              )
     

In [31]:
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ None, 1)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_46 (Conv2D)  │ (None, None,      │        160 │ input_layer_4[0]… │
│                     │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │         64 │ conv2d_46[0][0]   │
│ (BatchNormalizatio… │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_47 (Conv2D)  │ (None, None,      │      2,320 │ batch_normalizat… │
│                     │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │         64 │ conv2d_47[0][0]   │
│ (BatchNormalizatio… │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_8     │ (None, None,      │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_48 (Conv2D)  │ (None, None,      │      4,640 │ max_pooling2d_8[… │
│                     │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │        128 │ conv2d_48[0][0]   │
│ (BatchNormalizatio… │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_49 (Conv2D)  │ (None, None,      │      9,248 │ batch_normalizat… │
│                     │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │        128 │ conv2d_49[0][0]   │
│ (BatchNormalizatio… │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_9     │ (None, None,      │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_50 (Conv2D)  │ (None, None,      │     18,496 │ max_pooling2d_9[… │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │        256 │ conv2d_50[0][0]   │
│ (BatchNormalizatio… │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_51 (Conv2D)  │ (None, None,      │     36,928 │ batch_normalizat… │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, None,      │        256 │ conv2d_51[0][0]   │
│ (BatchNormalizatio… │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_10    │ (None, None,      │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_52 (Conv2D)  │ (None, None,      │     73,856 │ max_pooling2d_10

 Total params: 2,168,977 (8.27 MB)

 Trainable params: 2,165,489 (8.26 MB)

 Non-trainable params: 3,488 (13.62 KB)

In [32]:
batch_size = 5
n_epochs = 100
model.fit(x=[x_train_mix,x_train_label], y=y_train, batch_size = batch_size,
                    epochs = n_epochs, validation_data = ([x_valid_mix,x_valid_label], y_valid))

Epoch 1/100
48/48 ━━━━━━━━━━━━━━━━━━━━ 15s 149ms/step - loss: 0.0498 - mae: 0.1748 - val_loss: 0.0546 - val_mae: 0.1962
Epoch 2/100
48/48 ━━━━━━━━━━━━━━━━━━━━ 6s 134ms/step - loss: 0.0285 - mae: 0.1326 - val_loss: 0.0598 - val_mae: 0.1996
Epoch 3/100
48/48 ━━━━━━━━━━━━━━━━━━━━ 6s 131ms/step - loss: 0.0259 - mae: 0.1250 - val_loss: 0.0925 - val_mae: 0.2486
Epoch 4/100
48/48 ━━━━━━━━━━━━━━━━━━━━ 6s 131ms/step - loss: 0.0273 - mae: 0.1270 - val_loss: 0.0825 - val_mae: 0.2245
Epoch 5/100
48/48 ━━━━━━━━━━━━━━━━━━━━ 6s 132ms/step - loss: 0.0296 - mae: 0.1335 - val_loss: 0.0315 - val_mae: 0.1335
Epoch 6/100
48/48 ━━━━━━━━━━━━━━━━━━━━ 6s 132ms/step - loss: 0.0235 - mae: 0.1179 - val_loss: 0.0351 - val_mae: 0.1396
Epoch 7/100
48/48 ━━━━━━━━━━━━━━━━━━━━ 6s 133ms/step - loss: 0.0246 - mae: 0.1204 - val_loss: 0.0353 - val_mae: 0.1381
Epoch 8/100
48/48 ━━━━━━━━━━━━━━━━━━━━ 6s 132ms/step - loss: 0.0227 - mae: 0.1154 - val_loss: 0.0415 - val_mae: 0.1492
Epoch 9/100
48/48 ━━━━━━━━━━━━━━━━━━━━ 6s 131ms

In [14]:
gc.collect()

model.save('my_separator_model_prototype.keras')

In [36]:
img = cv2.imread("music_seperation_dataset/test/full_mix/53_full_mix.png",0)

img=cv2.resize(img,(128,128))
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
drum = np.array([[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,]])

predimg= np.squeeze(model.predict([img,drum]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_drum_part.png",prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
(128, 128)


True

In [37]:
import librosa

image = cv2.imread("separated_drum_part.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)

import soundfile
soundfile.write('drum_prediction.wav',audio, 22050)




In [40]:
img = cv2.imread("music_seperation_dataset/test/full_mix/53_full_mix.png",0)

img=cv2.resize(img,(128,128))
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
acoustic = np.array([[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,]])

predimg= np.squeeze(model.predict([img,acoustic]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part_acoustic.png",prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
(128, 128)


True

In [42]:
import librosa

image = cv2.imread("separated_part_acoustic.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)

import soundfile
soundfile.write('acoustic_prediction.wav',audio, 22050)

In [43]:
img = cv2.imread("music_seperation_dataset/test/full_mix/53_full_mix.png",0)

img=cv2.resize(img,(128,128))
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
bass = np.array([[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,]])

predimg= np.squeeze(model.predict([img,bass]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part_bass.png",prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
(128, 128)


True

In [44]:
import librosa

image = cv2.imread("separated_part_bass.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)

import soundfile
soundfile.write('bass_prediction.wav',audio, 22050)

In [45]:
img = cv2.imread("music_seperation_dataset/test/full_mix/53_full_mix.png",0)

img=cv2.resize(img,(128,128))
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
eletric = np.array([[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,]])

predimg= np.squeeze(model.predict([img,eletric]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part_eletric.png",prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
(128, 128)


True

In [46]:
import librosa

image = cv2.imread("separated_part_eletric.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)

import soundfile
soundfile.write('eletric_prediction.wav',audio, 22050)

In [47]:
img = cv2.imread("music_seperation_dataset/test/full_mix/53_full_mix.png",0)

img=cv2.resize(img,(128,128))
img= np.reshape(img,(-1, 128, 128, 1))
img=img/255
keyboard = np.array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,]])

predimg= np.squeeze(model.predict([img,keyboard]))

prediction= predimg*255
print(prediction.shape)

cv2.imwrite("separated_part_keyboard.png",prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
(128, 128)


True

In [49]:
import librosa

image = cv2.imread("separated_part_keyboard.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)

import soundfile
soundfile.write('keyboard_prediction.wav',audio, 22050)

In [ ]:
image = cv2.imread("music_seperation_dataset/test/full_mix/53_full_mix.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=256,sr=22050)


audio = librosa.util.normalize(audio)

import soundfile
soundfile.write('full.wav',audio, 22050)

In [ ]:
image = cv2.imread("music_seperation_dataset/test/Drum_set/53_Drum_set.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=1000,sr=22050)


audio = librosa.util.normalize(audio)

import soundfile
soundfile.write('drum_part.wav',audio, 22050)

In [ ]:
img = cv2.imread("music_dataset_spectro_full/valid/songsfull/0_songsfull.png",0)

img=cv2.resize(img,(256,6016))
img= np.reshape(img,(-1, 256, 6016, 1))
img=img/255

predimg= np.squeeze(model.predict(img))

prediction= predimg*255
print(prediction.shape)



cv2.imwrite("separated_part.png",prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 692ms/step
(256, 6016)


True

In [ ]:
import librosa

image = cv2.imread("music_dataset_spectro_full/valid/songsfull/0_songsfull.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=32,sr=22050,power=1)

audio = librosa.util.normalize(audio)

import soundfile
soundfile.write('song.wav',audio, 22050)

In [ ]:
freq, sr = librosa.load("music_dataset/songsfull/14.wav")

short_time= librosa.stft(freq)

mag,phase = librosa.magphase(short_time)

print(phase.shape)
print(mag.shape)
mel = librosa.feature.melspectrogram(sr=sr,S=mag)

inverse_mel = librosa.feature.inverse.mel_to_stft(mel,sr=sr)


short_inverse = mag * np.exp(phase)

recon = librosa.istft(short_inverse)
recon = librosa.util.normalize(recon)

soundfile.write('song.wav',recon, 22050)


(1025, 6014)
(1025, 6014)
(128, 6014)
